In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 23:50:13.710662: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 23:50:14.475650: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 23:50:15,534 [DEBUG] [Rain] Rain is initialized
2023-07-05 23:50:15,536 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 23:50:15,537 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 23:50:15,539 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 23:50:15,541 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-05 23:50:15,542 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 23:50:15,545 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 23:50:15,546 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 23:50:15,558 [INFO] [Provisioner] provisioner is serving
2023-07-05 23:50:15,558 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 23:50:15,560 [INFO] [Coordinator] coordinator is serving
2023-07-05 23:50:15,561 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 23:50:15,565 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 23:50:15,566 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 23:50:15,567 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-05 23:50:15,568 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-05 23:50:15,569 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 23:50:15,570 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50152/
2023-07-05 23:50:15,573 [INFO] [Worker_50152] Worker is running on port: 50152
2023-0

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 9ms/step - loss: 0.6948 - accuracy: 0.7824
Epoch 2/5
157/157 [==============================] - 3s 9ms/step - loss: 0.6914 - accuracy: 0.7831
Epoch 2/5
Epoch 2/5
157/157 [==============================] - 1s 7ms/step - loss: 0.3012 - accuracy: 0.9100
Epoch 3/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2990 - accuracy: 0.9084
Epoch 3/5
Epoch 3/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2308 - accuracy: 0.9313
Epoch 4/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2253 - accuracy: 0.9309
Epoch 4/5
157/157 [==============================] - 1s 7ms/step - loss: 0.2308 - accuracy: 0.9313
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1971 - accuracy: 0.9414
Epoch 5/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1847 - accuracy: 0.9437
Epoch 5/5
Epoch 5/5
157/157 [==============================] - 1s 8ms

2023-07-05 23:50:29,998 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1


sending data to divider


2023-07-05 23:50:30,002 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

157/157 [==============================] - 1s 8ms/step - loss: 0.0849 - accuracy: 0.9731


2023-07-05 23:50:46,630 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-05 23:50:46,631 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-05 23:50:46,642 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-05 23:50:46,651 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2
2023-07-05 23:50:46,679 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 2.
2023-07-05 23:50:49,678 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-05 23:50:49,680 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-05 23:50:49,689 [DEBUG

sending data to divider


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0884 - accuracy: 0.9761

Test accuracy: 97.6%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 23:50:50,221 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-05 23:50:50,222 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-05 23:50:50,224 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-05 23:50:50,225 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-05 23:50:50,227 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 23:50:50,229 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-05 23:50:50,230 [DEBUG] [LocalProvisioner] Creating 3 workers
DEBUG:LocalProvisioner:Creating 3 

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 8ms/step - loss: 0.1080 - accuracy: 0.9664
Epoch 2/5
157/157 [==============================] - 3s 8ms/step - loss: 0.1113 - accuracy: 0.9669
Epoch 2/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0955 - accuracy: 0.9701
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0843 - accuracy: 0.9737
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.0866 - accuracy: 0.9733
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.0791 - accuracy: 0.9752
Epoch 5/5
157/157 [==============================] - 1s 8ms/step - loss: 0.0705 - accuracy: 0.9778
Epoch 5/5
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0643 - accuracy: 0.9793


2023-07-05 23:51:00,408 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-05 23:51:00,411 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


sending data to divider
sending data to divider


DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
2023-07-05 23:51:00,415 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-05 23:51:00,417 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-05 23:51:00,421 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-05 23:51:00,423 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worke

...

2023-07-05 23:51:15,262 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully


157/157 [==============================] - 1s 6ms/step - loss: 0.0503 - accuracy: 0.9840


2023-07-05 23:51:15,289 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-05 23:51:15,291 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-05 23:51:15,292 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3


sending data to divider
sending data to divider


2023-07-05 23:51:15,293 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-05 23:51:15,306 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-05 23:51:15,307 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-05 23:51:15,329 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-05 23:51:15,331 [DEBUG] [DividerAmbassador] divider ambas

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0707 - accuracy: 0.9813

Test accuracy: 98.1%
